# Metode Baru: Validasi Viral, Estimasi Waktu Viral, dan Ekstraksi Multi-Tempat

Notebook ini adalah **lanjutan/pelengkap** dari `scrap_tempat_viral.ipynb` dan `scrap_tempat_viral_extend.ipynb`. Notebook-notebook itu tidak diubah — notebook ini memakai data yang sudah dikumpulkan (`df_news_extended_gabungan.csv`) dan mengganti/menambah langkah **ekstraksi & penilaian viral** untuk menjawab 3 kesulitan yang dialami:

| # | Kesulitan | Solusi di notebook ini |
| - | --- | --- |
| 1 | Validitas "viral" rendah — artikel sering hanya berisi tempat mainstream/rekomendasi, bukan yang benar-benar ramai dibahas | Skor `viral_confidence_score` gabungan dari **(a)** bahasa eksplisit "viral/ramai dibahas" di artikel (dinilai LLM, bukan regex), **(b)** jumlah *sumber berita unik* yang memberitakan tempat yang sama dalam rentang waktu pendek (proxy "ramai dibahas" tanpa perlu API media sosial berbayar), **(c)** lonjakan (spike) minat pencarian di **Google Trends** (gratis, via `pytrends`) untuk nama tempat tersebut |
| 2 | Tanggal mulai viral sulit dicari | `viral_since_estimate` dipilih berdasarkan prioritas: tanggal eksplisit yang disebut artikel → tanggal *spike* Google Trends → tanggal artikel paling awal dalam klaster pemberitaan. Kolom `viral_since_source` mencatat metode mana yang dipakai supaya transparan |
| 3 | Ekstraksi tempat dari artikel sering tidak lengkap (regex berbasis judul saja) | Ekstraksi diganti dengan **LLM (Claude) structured extraction** yang membaca seluruh isi artikel dan mengembalikan **daftar semua tempat** yang disebut (bukan cuma 1 dari judul), lengkap dengan lokasi, alasan viral, dan indikasi eksplisit/tidaknya viral. Ada fallback regex multi-tempat bila tidak ada API key |

Alur kerja:

1. Muat `df_news_extended_gabungan.csv` (tidak fetch ulang).
2. Ekstraksi multi-tempat per artikel (LLM, fallback regex).
3. Normalisasi & deduplikasi nama tempat antar-artikel (fuzzy matching) → satu baris per tempat unik.
4. Hitung sinyal viral: diversitas sumber, klaster waktu pemberitaan, Google Trends.
5. Hitung `viral_confidence_score` & `viral_since_estimate`, tandai kandidat yang "cuma mainstream" untuk direview manual.
6. Geocoding tempat yang lolos threshold (pakai fungsi yang sama seperti notebook utama).
7. Simpan hasil akhir + catatan keterbatasan.

## 1. Install & Import Library

Tambahan dari notebook utama: `anthropic` (LLM extraction), `rapidfuzz` (fuzzy dedup nama tempat).

In [ ]:
# %pip install pandas requests geopy pytrends rapidfuzz anthropic python-dateutil tqdm -q

Note: you may need to restart the kernel to use updated packages.


In [1]:
import glob
import json
import os
import re
import time
from datetime import datetime, timedelta

import pandas as pd
from dateutil import parser as dateparser
from rapidfuzz import fuzz
from tqdm.auto import tqdm

pd.set_option("display.max_colwidth", 120)

c:\Users\Agus Zainal Arifin\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1b. Gabungkan Batch Berita Menjadi Satu File

Menggabungkan beberapa file batch (`df_news_extended_255.csv`, `df_news_extended_256_456.csv`, `df_news_extended_456_656.csv`, `df_news_extended_706_906.csv`, `df_news_extended_full.csv`) menjadi satu file `df_news_extended_gabungan.csv`, dengan deduplikasi berdasarkan URL artikel.

In [2]:
MERGE_INPUT_FILES = [
    "df_news_extended_255.csv",
    "df_news_extended_256_456.csv",
    "df_news_extended_456_656.csv",
    "df_news_extended_706_906.csv",
    "df_news_extended_full.csv",
]
MERGE_OUTPUT_FILE = "df_news_extended_gabungan.csv"

merge_frames = []
for f in MERGE_INPUT_FILES:
    if not os.path.exists(f):
        print("File tidak ditemukan, dilewati:", f)
        continue
    df_part = pd.read_csv(f)
    print(f, "->", len(df_part), "baris")
    merge_frames.append(df_part)

df_merged = pd.concat(merge_frames, ignore_index=True)

merge_dedup_key = "final_url" if "final_url" in df_merged.columns else "link"
before = len(df_merged)
df_merged = df_merged.dropna(subset=[merge_dedup_key])
df_merged = df_merged.drop_duplicates(subset=[merge_dedup_key]).reset_index(drop=True)

print("Total sebelum dedup:", before)
print("Total setelah dedup:", len(df_merged))

df_merged.to_csv(MERGE_OUTPUT_FILE, index=False)
print("Tersimpan ke", MERGE_OUTPUT_FILE)

df_news_extended_255.csv -> 250 baris
df_news_extended_256_456.csv -> 184 baris
df_news_extended_456_656.csv -> 196 baris
df_news_extended_706_906.csv -> 196 baris
df_news_extended_full.csv -> 1656 baris
Total sebelum dedup: 2482
Total setelah dedup: 2285
Tersimpan ke df_news_extended_gabungan.csv


In [3]:
df_merged

,keyword,title,link,published,source,final_url,article_text,article_text_length
0,tempat makan viral,Media Sosial Mengubah Cara Kita Memilih Tempat Makan - Jatimnet.com,https://news.google.com/rss/articles/CBMifkFVX3lxTE9qZDRrbTc5elBjLU5ra0pZVnp1M21wMFRZd1NydUpGY2xlbm5vMktZZEJXNnhfcVZ...,2026-09-25 08:00:00+00:00,Jatimnet.com,https://jatimnet.com/media-sosial-mengubah-cara-kita-memilih-tempat-makan,"Jumat, 25 September 2026 08:00 UTC Ilustrasi: Pilihan dari Layar. -Dx Gen-AI JATIMNET.COM - Media sosial mengubah ca...",7110
1,tempat makan viral,Awas Kalap! Ini 7 Tempat Makan Soto Mie Legendaris di Jakarta dan Bogor - detikFood,https://news.google.com/rss/articles/CBMitgFBVV95cUxOYk92ckFweFFab1BZMHdPUTB2U2NXeE5vOVpHOUJfbXVqUm5YeHNHVjVUdUYxT2x...,2026-09-26 08:00:30+00:00,detikFood,https://food.detik.com/rumah-makan/d-8679688/awas-kalap-ini-7-tempat-makan-soto-mie-legendaris-di-jakarta-dan-bogor,"Di Jakarta dan Bogor, ada kedai soto mie yang sudah buka sejak puluhan tahun lalu dan terkenal lezat. Bahkan sampai ...",2855
2,tempat makan viral,Rekomendasi 7 Tempat Makan Ramah Kantong di Jogja dan Tips Menggunakan GrabFood - RRI.co.id,https://news.google.com/rss/articles/CBMiwgFBVV95cUxPQzJKTVJPcXJKa3ZnWnhCTXZUYVBySTd5d25iek4xRmdETXZXdWFicVJXdkl4ZGF...,2026-09-23 06:38:00+00:00,RRI.co.id,https://rri.co.id/yogyakarta/kuliner/2752639/rekomendasi-7-tempat-makan-ramah-kantong-di-jogja-dan-tips-menggunakan-...,"RRI.CO.ID, Yogyakarta - Daerah Istimewa Yogyakarta (DIY).dikenal luas sebagai surganya kuliner murah di Indonesia, t...",6348
3,tempat makan viral,10 Tempat Makan Viral di Bandung 2026 - nibble.id,https://news.google.com/rss/articles/CBMiaEFVX3lxTE9pZzU0UEt2M3VSYUxyTVBQdWtIMDEzaDhmRld1eWFGWk5ZaXc4Q0F0Q1FlQktpQW9...,2026-06-09 07:00:00+00:00,nibble.id,https://www.nibble.id/tempat-makan-viral-di-bandung-2026/,"Harus siapkan mental buat antre, ini dia tempat makan viral di Bandung 2026 yang sampai kini selalu gak ada habisnya...",9858
4,tempat makan viral,"TikTok Food Fest 2026 Hari Terakhir di GBK, Ada 40 Lebih Tenant hingga Kuliner Viral - Aktualita.co",https://news.google.com/rss/articles/CBMidEFVX3lxTE5xMXhUUkZOTGJJcWZ2VmgwdGgtcnJsZ3NnbklPWENaLWV0RTVpM2Njc25KNDhpOFZ...,2026-09-27 10:22:00+00:00,Aktualita.co,https://aktualita.co/ragam/tiktok-food-fest-2026-hari-terakhir-gbk,"English edition available TikTok Food Fest 2026 Hari Terakhir di GBK, Ada 40 Lebih Tenant hingga Kuliner Viral Jakar...",6105
...,...,...,...,...,...,...,...,...
2280,tempat belanja viral Medan,Sebuah video yang memperlihatkan perdebatan antara seorang pelanggan dan karyawan sebuah minimarket di Jalan Karya W...,https://news.google.com/rss/articles/CBMi3AFBVV95cUxNZzlaQzFZX3c4bk81MUhTUjh1UFA5Tzg0NXdueHctRUNyQi1SVnZUODVnS1VuNGZ...,2026-07-11 07:00:00+00:00,facebook.com,https://www.detik.com/jabar/berita/d-8677675/bukan-sekadar-beasiswa-genzi-ika-unpad-siapkan-calon-pemimpin,Program Beasiswa GENZI UPZ IKA Unpad hadir dengan fokus pada pembentukan karakter dan kepemimpinan. 50 mahasiswa ter...,158
2281,tempat belanja viral Medan,Sebuah video yang memperlihatkan keributan antara seorang driver ojek online (ojol) dengan karyawan sebuah toko vira...,https://news.google.com/rss/articles/CBMi3AFBVV95cUxNaVk2UUJWTFZhdEdYbmlpNFFKYjEyeVBtNWVJZWpjQ2lfYzNuVXpSVnJfTi0yOUU...,2026-07-09 07:00:00+00:00,facebook.com,https://batampos.jawapos.com/lifestyle/2606110016/5-tempat-makan-siang-enak-di-batam-yang-cocok-untuk-akhir-pekan-da...,Makan siang selalu menjadi momen krusial memilih makanan di Batam. Berikut 5 tempat makan siang enak di Batam.,110
2282,pasar viral Medan,"Diduga Belum Serah Terima, Pedagang 4 Pasar Dihimbau Bayar Kontibusi Kepada Wali Kota - Viral24.co.id",https://news.google.com/rss/articles/CBMisAFBVV95cUxQV3dfdVIyTkNERUpIbkVhNllsN2ZIOFhXQ0d4M19aOHJ4WVppUXJfMWg5V0hVVTR...,2026-09-24 07:33:28+00:00,Viral24.co.id,https://metropolis.batampos.co.id/jelajah-kuliner-batam-5-tempat-makan-baru-dengan-menu-unik-dan-instagramable/,batampos – Dunia kuliner Kota Batam teru

## 2. Muat Data Berita

Tidak fetch ulang — memakai `df_news_extended_gabungan.csv` (hasil gabungan beberapa batch `df_news_extended_*.csv`), buang duplikat berdasarkan `final_url`, dan buang artikel yang isinya terlalu pendek.

In [4]:
NEWS_CSV_FILE = "df_news_extended_gabungan.csv"
MIN_ARTICLE_LEN = 150

df_news_all = pd.read_csv(NEWS_CSV_FILE)
print("File dipakai:", NEWS_CSV_FILE)

# beberapa batch mungkin punya nama kolom isi artikel yang sedikit berbeda; standardisasi
text_col_candidates = [c for c in ["article_text", "article_text_extended", "text"] if c in df_news_all.columns]
if "article_text" not in df_news_all.columns and text_col_candidates:
    df_news_all["article_text"] = df_news_all[text_col_candidates[0]]

dedup_key = "final_url" if "final_url" in df_news_all.columns else "link"
df_news_all = df_news_all.dropna(subset=[dedup_key])
df_news_all = df_news_all.drop_duplicates(subset=[dedup_key]).reset_index(drop=True)
df_news_all = df_news_all[df_news_all["article_text"].fillna("").str.len() >= MIN_ARTICLE_LEN].reset_index(drop=True)

print("Total artikel unik & layak diproses:", len(df_news_all))
df_news_all.head(3)

File dipakai: df_news_extended_gabungan.csv
Total artikel unik & layak diproses: 2181


,keyword,title,link,published,source,final_url,article_text,article_text_length
0,tempat makan viral,Media Sosial Mengubah Cara Kita Memilih Tempat Makan - Jatimnet.com,https://news.google.com/rss/articles/CBMifkFVX3lxTE9qZDRrbTc5elBjLU5ra0pZVnp1M21wMFRZd1NydUpGY2xlbm5vMktZZEJXNnhfcVZ...,2026-09-25 08:00:00+00:00,Jatimnet.com,https://jatimnet.com/media-sosial-mengubah-cara-kita-memilih-tempat-makan,"Jumat, 25 September 2026 08:00 UTC Ilustrasi: Pilihan dari Layar. -Dx Gen-AI JATIMNET.COM - Media sosial mengubah ca...",7110
1,tempat makan viral,Awas Kalap! Ini 7 Tempat Makan Soto Mie Legendaris di Jakarta dan Bogor - detikFood,https://news.google.com/rss/articles/CBMitgFBVV95cUxOYk92ckFweFFab1BZMHdPUTB2U2NXeE5vOVpHOUJfbXVqUm5YeHNHVjVUdUYxT2x...,2026-09-26 08:00:30+00:00,detikFood,https://food.detik.com/rumah-makan/d-8679688/awas-kalap-ini-7-tempat-makan-soto-mie-legendaris-di-jakarta-dan-bogor,"Di Jakarta dan Bogor, ada kedai soto mie yang sudah buka sejak puluhan tahun lalu dan terkenal lezat. Bahkan sampai ...",2855
2,tempat makan viral,Rekomendasi 7 Tempat Makan Ramah Kantong di Jogja dan Tips Menggunakan GrabFood - RRI.co.id,https://news.google.com/rss/articles/CBMiwgFBVV95cUxPQzJKTVJPcXJKa3ZnWnhCTXZUYVBySTd5d25iek4xRmdETXZXdWFicVJXdkl4ZGF...,2026-09-23 06:38:00+00:00,RRI.co.id,https://rri.co.id/yogyakarta/kuliner/2752639/rekomendasi-7-tempat-makan-ramah-kantong-di-jogja-dan-tips-menggunakan-...,"RRI.CO.ID, Yogyakarta - Daerah Istimewa Yogyakarta (DIY).dikenal luas sebagai surganya kuliner murah di Indonesia, t...",6348


## 3. Ekstraksi Multi-Tempat per Artikel

**Kesulitan #3 (ekstraksi tidak lengkap)** diatasi dengan meminta LLM membaca seluruh isi artikel dan mengembalikan **semua** tempat yang disebut (bisa 1 artikel = banyak tempat, misal artikel "7 Kafe Viral di Bandung"), bukan cuma 1 tempat dari judul.

Skema output per tempat:

- `nama_tempat`
- `kategori` (Kafe/Resto/Wisata Alam/Museum/Wahana/Pameran/dll)
- `lokasi_text` — lokasi mentah seperti disebut artikel (kelurahan/kecamatan/kota/provinsi apa adanya)
- `alasan_viral` — ringkasan alasan/daya tarik
- `indikasi_waktu` — tanggal/bulan/periode eksplisit yang disebut terkait mulai viral (atau `null` jika tidak ada)
- `is_explicitly_viral` — **true** hanya jika artikel benar-benar menyatakan tempat ini viral/ramai dibahas/jadi perbincangan (bukan sekadar "rekomendasi"/"wajib coba")
- `confidence` — 0-1, seberapa yakin LLM dengan hasil ekstraksi ini

Butuh `ANTHROPIC_API_KEY` di environment variable. **Tanpa API key, notebook tetap berjalan** memakai fallback regex multi-tempat (kurang akurat, tapi tidak butuh biaya) — lihat Bagian 3b.

In [ ]:
LLM_MODEL = "claude-sonnet-5"
USE_LLM = bool(os.environ.get("ANTHROPIC_API_KEY"))
print("Menggunakan LLM extraction:", USE_LLM)

EXTRACT_TOOL_SCHEMA = {
    "name": "extract_places",
    "description": "Ekstrak semua tempat makan/wisata/hiburan yang disebut dalam artikel berita, beserta indikasi viralnya.",
    "input_schema": {
        "type": "object",
        "properties": {
            "places": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "nama_tempat": {"type": "string"},
                        "kategori": {"type": "string", "description": "Kafe/Restoran/Wisata Alam/Museum/Wahana/Pameran/lainnya"},
                        "lokasi_text": {"type": "string", "description": "Lokasi mentah apa adanya sesuai artikel (kelurahan/kecamatan/kota/kabupaten/provinsi jika disebut)"},
                        "kelurahan_desa": {"type": ["string", "null"], "description": "Nama kelurahan/desa, null jika tidak disebut/tidak diketahui"},
                        "kecamatan": {"type": ["string", "null"], "description": "Nama kecamatan, null jika tidak disebut/tidak diketahui"},
                        "kota_kabupaten": {"type": ["string", "null"], "description": "Nama kota atau kabupaten, null jika tidak disebut/tidak diketahui"},
                        "provinsi": {"type": ["string", "null"], "description": "Nama provinsi, null jika tidak disebut/tidak diketahui"},
                        "alasan_viral": {"type": "string"},
                        "indikasi_waktu": {"type": ["string", "null"], "description": "Tanggal/bulan/periode eksplisit terkait mulai viral, atau null jika tidak disebut"},
                        "is_explicitly_viral": {"type": "boolean", "description": "true hanya jika artikel eksplisit menyebut tempat INI sedang viral/ramai dibahas/jadi perbincangan warganet, bukan sekadar direkomendasikan"},
                        "confidence": {"type": "number"},
                    },
                    "required": ["nama_tempat", "kategori", "lokasi_text", "alasan_viral", "is_explicitly_viral", "confidence"],
                },
            }
        },
        "required": ["places"],
    },
}

if USE_LLM:
    import anthropic
    client = anthropic.Anthropic()


def extract_places_llm(title: str, article_text: str, max_chars: int = 6000):
    """Ekstrak daftar tempat dari 1 artikel memakai Claude, hasil terstruktur lewat tool use."""
    text = article_text[:max_chars]
    prompt = (
        f"Judul artikel: {title}\n\nIsi artikel:\n{text}\n\n"
        "Ekstrak semua tempat makan/kafe/resto/wisata/museum/wahana/pameran yang disebutkan "
        "memakai tool extract_places. Selain `lokasi_text` (kutipan lokasi apa adanya), isi juga "
        "`kelurahan_desa`, `kecamatan`, `kota_kabupaten`, `provinsi` sebagai kolom terpisah jika "
        "informasinya ada/bisa disimpulkan dari artikel; isi null jika benar-benar tidak diketahui. "
        "Jika artikel tidak menyebut tempat spesifik, kembalikan places kosong."
    )
    resp = client.messages.create(
        model=LLM_MODEL,
        max_tokens=2000,
        tools=[EXTRACT_TOOL_SCHEMA],
        tool_choice={"type": "tool", "name": "extract_places"},
        messages=[{"role": "user", "content": prompt}],
    )
    for block in resp.content:
        if block.type == "tool_use" and block.name == "extract_places":
            return block.input.get("places", [])
    return []

### 3b. Fallback Tanpa LLM (Regex Multi-Tempat)

Kalau tidak ada `ANTHROPIC_API_KEY`, tetap coba ambil **lebih dari 1 tempat per artikel** (bukan cuma dari judul seperti notebook lama) dengan mencari pola daftar bernomor/list (`1. Nama Tempat`, `- Nama Tempat`) yang biasa dipakai artikel "Rekomendasi N tempat viral", ditambah judul sebagai kandidat tambahan. Ini tetap heuristik dan kurang akurat dibanding LLM — dianggap sebagai *best-effort fallback*, bukan solusi utama.

In [ ]:
LIST_ITEM_PATTERN = re.compile(r"(?:^|\n)\s*(?:\d{1,2}[\.\)]|[-•])\s*([A-Z][\w'&\.\-]*(?:\s+[A-Z\w'&\.\-]+){0,5})", re.MULTILINE)
VIRAL_WORDS = re.compile(r"viral|ramai\s+dibahas|jadi\s+perbincangan|heboh|trending", re.IGNORECASE)
MONTH_NAMES = "januari|februari|maret|april|mei|juni|juli|agustus|september|oktober|november|desember"
DATE_PATTERN = re.compile(rf"(sejak\s+)?(\d{{1,2}}\s+)?({MONTH_NAMES})\s+\d{{4}}", re.IGNORECASE)


def extract_places_fallback(title: str, article_text: str):
    candidates = {}
    for m in LIST_ITEM_PATTERN.finditer(article_text):
        name = m.group(1).strip(" .")
        if 2 <= len(name.split()) <= 6 and name not in candidates:
            candidates[name] = None
    if not candidates:
        # fallback ke judul, konsisten dengan notebook utama
        title_name = re.split(r"\s+-\s+|\s+di\s+", title)[0].strip()
        candidates[title_name] = None

    date_match = DATE_PATTERN.search(article_text)
    indikasi_waktu = date_match.group(0) if date_match else None
    is_viral = bool(VIRAL_WORDS.search(article_text))

    places = []
    for name in candidates:
        places.append({
            "nama_tempat": name,
            "kategori": "tidak diketahui",
            "lokasi_text": "",
            "kelurahan_desa": None,
            "kecamatan": None,
            "kota_kabupaten": None,
            "provinsi": None,
            "alasan_viral": "",
            "indikasi_waktu": indikasi_waktu,
            "is_explicitly_viral": is_viral,
            "confidence": 0.3,
        })
    return places

### 3c. Alternatif: Ekstraksi Manual via Chat Claude.ai (Tanpa API Key)

Kalau kamu sudah punya **Claude Pro** tapi belum punya **Anthropic API key** (dua hal ini terpisah — Pro tidak otomatis memberi akses API), kamu tetap bisa memakai kualitas ekstraksi LLM dengan cara manual: generate prompt per-batch artikel di sini, tempel ke chat Claude.ai, lalu tempel balik hasil JSON-nya.

Langkahnya:

1. Jalankan sel di bawah — ini membuat file `manual_llm_batches/batch_001_prompt.txt`, `batch_002_prompt.txt`, dst (tiap file berisi beberapa artikel + instruksi format JSON).
2. Buka **chat baru** di claude.ai, salin **seluruh isi** satu file `batch_XXX_prompt.txt`, tempel sebagai pesan, kirim.
3. Salin **balasan JSON** dari Claude (pastikan hanya JSON, tanpa teks tambahan), simpan sebagai `manual_llm_batches/batch_XXX_response.json` (nama harus cocok dengan nomor batch-nya).
4. Ulangi untuk semua batch, lalu jalankan sel "Muat Hasil Manual ke Checkpoint" di bawah.
5. Setelah itu, **lanjut ke Bagian 4** langsung ke sel pembacaan checkpoint (skip sel loop otomatis) untuk membangun `df_places_raw`.

Jika artikelnya banyak, proses ini memang manual dan makan waktu — cocok untuk memvalidasi kualitas LLM dulu di sebagian data sebelum memutuskan berlangganan API.

In [ ]:
MANUAL_BATCH_DIR = "manual_llm_batches"
MANUAL_BATCH_SIZE = 8       # jumlah artikel per file prompt - kecilkan jika artikel panjang
MANUAL_MAX_CHARS = 3000     # potong isi artikel supaya prompt tidak terlalu panjang untuk chat

os.makedirs(MANUAL_BATCH_DIR, exist_ok=True)

# simpan pemetaan article_id -> metadata artikel, dipakai lagi saat mengimpor hasil balasan
id_map = {}
for idx, row in df_news_all.iterrows():
    id_map[str(idx)] = {
        "final_url": row[dedup_key],
        "title": row.get("title", ""),
        "source": row.get("source", ""),
        "published": row.get("published", ""),
    }
with open(os.path.join(MANUAL_BATCH_DIR, "id_map.json"), "w", encoding="utf-8") as f:
    json.dump(id_map, f, ensure_ascii=False, indent=2)

PROMPT_HEADER = """Kamu akan menerima beberapa artikel berita. Untuk SETIAP artikel, ekstrak semua tempat makan/kafe/resto/wisata/museum/wahana/pameran yang disebutkan.

Balas HANYA dengan JSON valid (tanpa penjelasan, tanpa markdown code fence) dengan format:
{
  "results": [
    {
      "article_id": <id artikel, angka>,
      "places": [
        {
          "nama_tempat": "...",
          "kategori": "Kafe/Restoran/Wisata Alam/Museum/Wahana/Pameran/lainnya",
          "lokasi_text": "lokasi mentah apa adanya sesuai artikel atau kamu cari di internet (kelurahan/kecamatan/kota/kabupaten/provinsi)",
          "kelurahan_desa": "nama kelurahan/desa, atau null jika tidak diketahui",
          "kecamatan": "nama kecamatan, atau null jika tidak diketahui",
          "kota_kabupaten": "nama kota/kabupaten, atau null jika tidak diketahui",
          "provinsi": "nama provinsi, atau null jika tidak diketahui",
          "alasan_viral": "ringkasan alasan/daya tarik",
          "indikasi_waktu": "tanggal/bulan/periode eksplisit terkait mulai viral, atau null jika tidak disebut",
          "is_explicitly_viral": true atau false (true HANYA jika artikel eksplisit menyebut tempat ini viral/ramai dibahas/jadi perbincangan warganet, bukan sekadar direkomendasikan),
          "confidence": angka 0-1
        }
      ]
    }
  ]
}

Jika sebuah artikel tidak menyebut tempat spesifik, kembalikan places: [] untuk artikel itu. Sertakan SEMUA article_id yang diberikan di bawah, jangan ada yang terlewat.
"""

article_ids = list(df_news_all.index)
batches = [article_ids[i:i + MANUAL_BATCH_SIZE] for i in range(0, len(article_ids), MANUAL_BATCH_SIZE)]

for b_num, batch_ids in enumerate(batches, start=1):
    parts = [PROMPT_HEADER, "\nArtikel:\n"]
    for aid in batch_ids:
        row = df_news_all.loc[aid]
        text = str(row.get("article_text", ""))[:MANUAL_MAX_CHARS]
        parts.append(f"\n### Artikel {aid}\nJudul: {row.get('title', '')}\nIsi:\n{text}\n")
    prompt_text = "\n".join(parts)
    fname = os.path.join(MANUAL_BATCH_DIR, f"batch_{b_num:03d}_prompt.txt")
    with open(fname, "w", encoding="utf-8") as f:
        f.write(prompt_text)

print(f"Dibuat {len(batches)} file prompt di folder '{MANUAL_BATCH_DIR}/'")
print("Total artikel:", len(article_ids))
print("Contoh file:", os.path.join(MANUAL_BATCH_DIR, "batch_001_prompt.txt"))

Dibuat 273 file prompt di folder 'manual_llm_batches/'
Total artikel: 2181
Contoh file: manual_llm_batches\batch_001_prompt.txt


**Setelah semua file `batch_XXX_response.json` tersimpan** di folder `responses_json_73_batch/` (isi setiap file harus JSON valid seperti contoh format di atas), jalankan sel berikut untuk memuat semuanya ke `places_raw_checkpoint.jsonl` — file checkpoint yang sama dipakai Bagian 4, jadi kamu bisa langsung lanjut ke sel pembacaan checkpoint di Bagian 4 tanpa menjalankan sel loop otomatis (Opsi A/B) sama sekali.

In [7]:
MANUAL_BATCH_DIR = "manual_llm_batches"
RESPONSE_DIR = "responses_json_73_batch"  # folder tempat batch_XXX_response.json disimpan
CHECKPOINT_FILE = "places_raw_checkpoint.jsonl"  # sama dengan konstanta di Bagian 4

response_files = sorted(glob.glob(os.path.join(RESPONSE_DIR, "batch_*_response.json")))
print("File respons ditemukan:", len(response_files))

with open(os.path.join(MANUAL_BATCH_DIR, "id_map.json"), encoding="utf-8") as f:
    id_map = json.load(f)

already_done_manual = set()
if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE, encoding="utf-8") as f:
        for line in f:
            try:
                already_done_manual.add(json.loads(line)["_article_key"])
            except Exception:
                continue

written = 0
with open(CHECKPOINT_FILE, "a", encoding="utf-8") as out_f:
    for rf in response_files:
        with open(rf, encoding="utf-8") as f:
            try:
                data = json.load(f)
            except json.JSONDecodeError as e:
                print("Gagal parse", rf, "-> pastikan isinya JSON valid tanpa teks tambahan:", e)
                continue
        for result in data.get("results", []):
            aid = str(result.get("article_id"))
            meta = id_map.get(aid)
            if not meta:
                print("article_id tidak dikenal:", aid, "di", rf)
                continue
            key = meta["final_url"]
            if key in already_done_manual:
                continue
            places = result.get("places", [])
            for p in places:
                p["_article_key"] = key
                p["title"] = meta["title"]
                p["source"] = meta["source"]
                p["published"] = meta["published"]
                p["final_url"] = key
                out_f.write(json.dumps(p, ensure_ascii=False) + "\n")
                written += 1
            if not places:
                out_f.write(json.dumps({"_article_key": key, "nama_tempat": None}, ensure_ascii=False) + "\n")
            already_done_manual.add(key)

print("Baris ditambahkan ke checkpoint:", written)
print("Lanjut ke sel pembacaan checkpoint di Bagian 4 (yang membangun df_places_raw).")

File respons ditemukan: 73
Baris ditambahkan ke checkpoint: 1788
Lanjut ke sel pembacaan checkpoint di Bagian 4 (yang membangun df_places_raw).


## 4. Jalankan Ekstraksi ke Semua Artikel (Opsi A: API / Opsi B: Regex Otomatis)

**Lewati bagian ini jika kamu memakai Opsi C (manual via chat Claude.ai) di Bagian 3c** — checkpoint sudah terisi dari sana, langsung lompat ke sel pembacaan checkpoint di bawah.

Sel di bawah menjalankan ekstraksi otomatis (LLM via API jika `ANTHROPIC_API_KEY` tersedia, atau fallback regex jika tidak). Setiap baris hasil = 1 tempat yang disebut di 1 artikel (`df_places_raw`). Progres disimpan berkala ke `places_raw_checkpoint.jsonl` supaya aman jika proses terhenti di tengah jalan (relevan terutama untuk pemanggilan LLM berbayar).

In [5]:
CHECKPOINT_FILE = "places_raw_checkpoint.jsonl"
REQUEST_DELAY = 0.3

already_done = set()
if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE, encoding="utf-8") as f:
        for line in f:
            try:
                row = json.loads(line)
                already_done.add(row["_article_key"])
            except Exception:
                continue
print("Artikel yang sudah diproses sebelumnya:", len(already_done))

with open(CHECKPOINT_FILE, "a", encoding="utf-8") as out_f:
    for _, row in tqdm(df_news_all.iterrows(), total=len(df_news_all)):
        key = row[dedup_key]
        if key in already_done:
            continue
        try:
            if USE_LLM:
                places = extract_places_llm(row.get("title", ""), row.get("article_text", ""))
            else:
                places = extract_places_fallback(row.get("title", ""), row.get("article_text", ""))
        except Exception as e:
            places = []
            print("Gagal ekstraksi:", key, "->", e)

        for p in places:
            p["_article_key"] = key
            p["title"] = row.get("title", "")
            p["source"] = row.get("source", "")
            p["published"] = row.get("published", "")
            p["final_url"] = row.get(dedup_key, "")
            out_f.write(json.dumps(p, ensure_ascii=False) + "\n")
        if not places:
            out_f.write(json.dumps({"_article_key": key, "nama_tempat": None}, ensure_ascii=False) + "\n")
        out_f.flush()
        if USE_LLM:
            time.sleep(REQUEST_DELAY)

print("Selesai. Baca ulang checkpoint untuk lanjut ke langkah berikutnya.")

Artikel yang sudah diproses sebelumnya: 2181


100%|██████████| 2181/2181 [00:00<00:00, 25651.76it/s]

Selesai. Baca ulang checkpoint untuk lanjut ke langkah berikutnya.


In [8]:
rows = []
with open(CHECKPOINT_FILE, encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)
        if row.get("nama_tempat"):
            rows.append(row)

df_places_raw = pd.DataFrame(rows).drop_duplicates(subset=["_article_key", "nama_tempat"]).reset_index(drop=True)
print("Total baris tempat mentah (sebelum dedup antar-artikel):", len(df_places_raw))
df_places_raw.head(5)

Total baris tempat mentah (sebelum dedup antar-artikel): 1788


,nama_tempat,kategori,lokasi_text,alasan_viral,indikasi_waktu,is_explicitly_viral,confidence,_article_key,title,source,published,final_url
0,Soto Mie Agih,Restoran,"Jalan Suryakencana No.313, Babakan Pasar, Bogor Tengah, Bogor","Soto mie legendaris Bogor sejak 1965-an, menu babat hingga jumbo",None,False,0.90,https://food.detik.com/rumah-makan/d-8679688/awas-kalap-ini-7-tempat-makan-soto-mie-legendaris-di-jakarta-dan-bogor,Awas Kalap! Ini 7 Tempat Makan Soto Mie Legendaris di Jakarta dan Bogor - detikFood,detikFood,2026-09-26 08:00:30+00:00,https://food.detik.com/rumah-makan/d-8679688/awas-kalap-ini-7-tempat-makan-soto-mie-legendaris-di-jakarta-dan-bogor
1,Soto Mie Sarodja,Restoran,"Jalan Danau Tondano No.27, Pejompongan, Jakarta Pusat","Soto mie sejak 1970, diteruskan generasi kedua",None,False,0.90,https://food.detik.com/rumah-makan/d-8679688/awas-kalap-ini-7-tempat-makan-soto-mie-legendaris-di-jakarta-dan-bogor,Awas Kalap! Ini 7 Tempat Makan Soto Mie Legendaris di Jakarta dan Bogor - detikFood,detikFood,2026-09-26 08:00:30+00:00,https://food.detik.com/rumah-makan/d-8679688/awas-kalap-ini-7-tempat-makan-soto-mie-legendaris-di-jakarta-dan-bogor
2,Soto Mie Theresia,Restoran,"Jalan Gereja Theresia No.43, Menteng, Jakarta Pusat",Soto mie legendaris sejak 1972 di kawasan Kuliner Theresia,None,False,0.90,https://food.detik.com/rumah-makan/d-8679688/awas-kalap-ini-7-tempat-makan-soto-mie-legendaris-di-jakarta-dan-bogor,Awas Kalap! Ini 7 Tempat Makan Soto Mie Legendaris di Jakarta dan Bogor - detikFood,detikFood,2026-09-26 08:00:30+00:00,https://food.detik.com/rumah-makan/d-8679688/awas-kalap-ini-7-tempat-makan-soto-mie-legendaris-di-jakarta-dan-bogor
3,Soto Mie Pak Fauzy,Restoran,"Kawasan Suryakencana, Bogor",Soto mie legendaris sejak 1974,None,False,0.85,https://food.detik.com/rumah-makan/d-8679688/awas-kalap-ini-7-tempat-makan-soto-mie-legendaris-di-jakarta-dan-bogor,Awas Kalap! Ini 7 Tempat Makan Soto Mie Legendaris di Jakarta dan Bogor - detikFood,detikFood,2026-09-26 08:00:30+00:00,https://food.detik.com/rumah-makan/d-8679688/awas-kalap-ini-7-tempat-makan-soto-mie-legendaris-di-jakarta-dan-bogor
4,Gudeg Yu Djum,Restoran,"Jl. Wijilan No. 167, Panembahan, Kraton, Yogyakarta","Ikon gudeg Jogja sejak 1950-an, harga ekonomis",None,False,0.90,https://rri.co.id/yogyakarta/kuliner/2752639/rekomendasi-7-tempat-makan-ramah-kantong-di-jogja-dan-tips-menggunakan-...,Rekomendasi 7 Tempat Makan Ramah Kantong di Jogja dan Tips Menggunakan GrabFood - RRI.co.id,RRI.co.id,2026-09-23 06:38:00+00:00,https://rri.co.id/yogyakarta/kuliner/2752639/rekomendasi-7-tempat-makan-ramah-kantong-di-jogja-dan-tips-menggunakan-...


## 5. Normalisasi & Deduplikasi Nama Tempat (Fuzzy Matching)

Tempat yang sama bisa disebut dengan penulisan sedikit berbeda antar-artikel (mis. "Soto Mie Agih" vs "Soto Mie Agih Bogor"). Sebelum agregasi, normalisasi nama (lower-case, buang tanda baca berlebih) lalu **cluster** memakai `rapidfuzz.fuzz.token_sort_ratio` — nama dengan kemiripan di atas `FUZZY_THRESHOLD` dianggap tempat yang sama dan diberi `canonical_group` yang sama.

In [ ]:
FUZZY_THRESHOLD = 88  # 0-100, makin tinggi makin ketat kemiripan nama


def normalize_place_name(name: str) -> str:
    name = str(name).lower().strip()
    name = re.sub(r"[^\w\s]", " ", name)
    name = re.sub(r"\s+", " ", name).strip()
    return name


df_places_raw["_nama_norm"] = df_places_raw["nama_tempat"].map(normalize_place_name)

# cluster nama unik yang mirip (greedy): tiap nama unik dibandingkan ke representative cluster yang sudah ada
unique_names = df_places_raw["_nama_norm"].unique().tolist()
name_to_group = {}
cluster_reps = []  # representative nama tiap cluster, indexnya = id cluster

for name in unique_names:
    if not name:
        continue
    best_idx, best_score = None, 0
    for idx, rep in enumerate(cluster_reps):
        score = fuzz.token_sort_ratio(name, rep)
        if score > best_score:
            best_idx, best_score = idx, score
    if best_score >= FUZZY_THRESHOLD:
        name_to_group[name] = best_idx
    else:
        cluster_reps.append(name)
        name_to_group[name] = len(cluster_reps) - 1

df_places_raw["canonical_group"] = df_places_raw["_nama_norm"].map(name_to_group)

print("Total nama mentah unik:", len(unique_names))
print("Total canonical_group setelah fuzzy dedup:", df_places_raw["canonical_group"].nunique())

Total nama mentah unik: 1554
Total canonical_group setelah fuzzy dedup: 1535


### 5b. Perbandingan: Fuzzy Matching Nama Saja vs Nama + Lokasi

Clustering di atas (`canonical_group`) cuma membandingkan `nama_tempat`. Risikonya: nama generik seperti "Kafe Senja" atau "Warung Bu Siti" di kota yang berbeda bisa salah digabung jadi satu tempat padahal itu tempat yang berbeda.

Cell ini membuat clustering **kedua** yang membandingkan **nama + lokasi** (`lokasi_text`, fallback ke `kelurahan_desa`/`kecamatan`/`kota_kabupaten`/`provinsi` kalau `lokasi_text` kosong), lalu membandingkan hasilnya dengan clustering nama-saja lewat dua DataFrame terpisah:

- `df_places_name_only` — agregasi pakai `canonical_group` (nama saja, threshold `NAME_ONLY_THRESHOLD`)
- `df_places_combined` — agregasi pakai `canonical_group_combined` (nama + lokasi, threshold `COMBINED_THRESHOLD`)

Threshold-nya sengaja dibedakan (lihat sweep di bawah): skor gabungan (nama 70% + lokasi 30%) secara alami lebih rendah/berfluktuasi dibanding skor nama-saja karena kualitas `lokasi_text` tidak konsisten (kadang lengkap, kadang cuma nama kota), jadi threshold gabungan perlu sedikit lebih longgar supaya tempat yang sama tidak malah terpecah.

In [ ]:
# --- Skor gabungan nama + lokasi ---
NAME_ONLY_THRESHOLD = FUZZY_THRESHOLD  # dari cell sebelumnya (88) - dipakai ulang sebagai baseline
NAME_WEIGHT = 0.7
LOC_WEIGHT = 0.3


def build_location_text(row):
    loc_text = row.get("lokasi_text")
    if isinstance(loc_text, str) and loc_text.strip():
        return loc_text
    extra = [
        row.get(c) for c in ["kelurahan_desa", "kecamatan", "kota_kabupaten", "provinsi"]
        if isinstance(row.get(c), str) and row.get(c).strip()
    ]
    return ", ".join(extra) if extra else ""


df_places_raw["_lokasi_norm"] = df_places_raw.apply(build_location_text, axis=1).map(normalize_place_name)


def combined_score(item_a, item_b):
    name_a, loc_a = item_a
    name_b, loc_b = item_b
    name_score = fuzz.token_sort_ratio(name_a, name_b)
    if loc_a and loc_b:
        loc_score = fuzz.token_set_ratio(loc_a, loc_b)
        return NAME_WEIGHT * name_score + LOC_WEIGHT * loc_score
    return name_score  # salah satu/keduanya tanpa lokasi -> tidak bisa dibedakan, pakai skor nama saja


def cluster_by_score(unique_items, score_fn, threshold):
    reps = []
    item_to_group = {}
    for item in unique_items:
        best_idx, best_score = None, -1
        for idx, rep in enumerate(reps):
            score = score_fn(item, rep)
            if score > best_score:
                best_idx, best_score = idx, score
        if best_score >= threshold:
            item_to_group[item] = best_idx
        else:
            reps.append(item)
            item_to_group[item] = len(reps) - 1
    return item_to_group


# --- Sweep threshold untuk bantu pilih nilai optimal masing-masing pendekatan ---
unique_names = [n for n in df_places_raw["_nama_norm"].unique().tolist() if n]
unique_combo = [
    tuple(x) for x in df_places_raw[["_nama_norm", "_lokasi_norm"]]
    .drop_duplicates().itertuples(index=False, name=None) if x[0]
]

print("Sweep threshold - NAMA SAJA (fuzz.token_sort_ratio):")
for t in [80, 85, 88, 90, 95]:
    n_groups = len(set(cluster_by_score(unique_names, lambda a, b: fuzz.token_sort_ratio(a, b), t).values()))
    print(f"  threshold={t:>3} -> {n_groups} grup (dari {len(unique_names)} nama unik)")

print("\nSweep threshold - NAMA + LOKASI (skor gabungan):")
for t in [70, 75, 80, 82, 85, 88, 90]:
    n_groups = len(set(cluster_by_score(unique_combo, combined_score, t).values()))
    print(f"  threshold={t:>3} -> {n_groups} grup (dari {len(unique_combo)} kombinasi nama+lokasi unik)")

# Dipilih dari hasil sweep: titik di mana jumlah grup mulai landai (tidak lagi turun tajam)
# menandakan penambahan threshold tidak lagi menggabungkan pasangan yang jelas beda tempat.
COMBINED_THRESHOLD = 85

name_to_group = cluster_by_score(unique_names, lambda a, b: fuzz.token_sort_ratio(a, b), NAME_ONLY_THRESHOLD)
combo_to_group = cluster_by_score(unique_combo, combined_score, COMBINED_THRESHOLD)

df_places_raw["canonical_group_combined"] = df_places_raw.apply(
    lambda r: combo_to_group.get((r["_nama_norm"], r["_lokasi_norm"])), axis=1
)

print(f"\nTotal canonical_group (nama saja, threshold={NAME_ONLY_THRESHOLD}):", df_places_raw["canonical_group"].nunique())
print(f"Total canonical_group_combined (nama+lokasi, threshold={COMBINED_THRESHOLD}):", df_places_raw["canonical_group_combined"].nunique())


# --- Agregasi terpisah untuk tiap pendekatan, hasilnya di variabel df berbeda ---
def aggregate_places(df_raw, group_col):
    agg_rows = []
    for group, sub in df_raw.groupby(group_col):
        agg_rows.append({
            group_col: group,
            "nama_tempat": sub["nama_tempat"].mode().iat[0] if not sub["nama_tempat"].mode().empty else sub["nama_tempat"].iat[0],
            "kategori": sub["kategori"].mode().iat[0] if not sub["kategori"].mode().empty else "",
            "lokasi_text": next((l for l in sub["lokasi_text"] if isinstance(l, str) and l.strip()), ""),
            "jumlah_artikel": len(sub),
            "jumlah_source_unik": sub["source"].nunique(),
            "alasan_viral_contoh": next((a for a in sub["alasan_viral"] if isinstance(a, str) and a.strip()), ""),
        })
    return pd.DataFrame(agg_rows).sort_values("jumlah_source_unik", ascending=False).reset_index(drop=True)


df_places_name_only = aggregate_places(df_places_raw, "canonical_group")
df_places_combined = aggregate_places(df_places_raw, "canonical_group_combined")

print(f"\ndf_places_name_only: {len(df_places_name_only)} tempat unik")
print(f"df_places_combined : {len(df_places_combined)} tempat unik")

# Contoh kasus yang paling kelihatan bedanya: nama sama tapi grup nama-saja pecah/gabung beda dgn grup gabungan
compare = (
    df_places_raw.groupby("nama_tempat")
    .agg(n_group_nama=("canonical_group", "nunique"), n_group_gabungan=("canonical_group_combined", "nunique"))
)
diff_cases = compare[compare["n_group_nama"] != compare["n_group_gabungan"]]
print(f"\nJumlah nama_tempat yang hasil clustering-nya beda antara nama-saja vs nama+lokasi: {len(diff_cases)}")
diff_cases.head(10)

In [16]:
cluster_reps

['soto mie agih',
 'soto mie sarodja',
 'soto mie theresia',
 'soto mie pak fauzy',
 'gudeg yu djum',
 'gudeg pawon',
 'rm abah harja',
 'rm wangi seruni',
 'tiktok food fest 2026',
 'mad bagel',
 'rm lokiin',
 'chuchat',
 'fuku',
 'eight tarts',
 'ayam renald',
 'aldi s burger',
 'sop janda',
 'bebek kaleyo',
 'fish n chops',
 'ayam goreng kalibogor',
 'warung amanda',
 'warung sego babat jalan diponegoro',
 'riung tenda',
 'sari idaman',
 'lesehan gurame pemuda',
 'sambal bakar indonesia',
 'teras rumah nenek',
 'bakso president',
 'rawon nguling',
 'depot hok lay',
 'laksa pak masitar',
 'warung bogana may may',
 'harmadha joglo',
 'rm mang iki',
 'rm saung cikenceh',
 'ayam bakar christina abc',
 'rumah makan gandy',
 'nasi uduk h yoyo',
 'rm salam sereh',
 'the layer cafe',
 'padre',
 'two stories',
 'kopi nako',
 'sate madura cak mat',
 'ayam bakar madu',
 'kampung kecil km 5',
 'le garden',
 'bmb canteen',
 'toko bubur dibawah pohon rindang',
 'pasar cihapit',
 'malindo',
 'momo

## 6. Agregasi Sinyal per Tempat

Untuk setiap tempat unik, hitung:

- `jumlah_artikel`, `jumlah_source_unik` — proxy "banyak dibahas" tanpa API media sosial
- `rentang_hari_pemberitaan` — makin pendek rentang tanggal publikasi antar-artikel yang membahasnya, makin kuat indikasi *lonjakan* pemberitaan (bukan cuma tempat lama yang berulang-ulang direkomendasikan selama bertahun-tahun)
- `explicit_viral_ratio` — proporsi artikel yang eksplisit bilang tempat ini viral/ramai dibahas (bukan sekadar direkomendasikan)
- `earliest_article_date`, `indikasi_waktu_llm` (tanggal eksplisit tercatat dari LLM/regex, jika ada)

In [10]:
def safe_parse_date(x):
    try:
        return dateparser.parse(str(x))
    except Exception:
        return None


df_places_raw["_pub_date"] = df_places_raw["published"].map(safe_parse_date)

agg_rows = []
for group, sub in df_places_raw.groupby("canonical_group"):
    dates = sub["_pub_date"].dropna()
    explicit_dates = sub["indikasi_waktu"].dropna()
    agg_rows.append({
        "canonical_group": group,
        "nama_tempat": sub["nama_tempat"].mode().iat[0] if not sub["nama_tempat"].mode().empty else sub["nama_tempat"].iat[0],
        "kategori": sub["kategori"].mode().iat[0] if not sub["kategori"].mode().empty else "",
        "lokasi_text": next((l for l in sub["lokasi_text"] if isinstance(l, str) and l.strip()), ""),
        "kelurahan_desa": next((v for v in sub.get("kelurahan_desa", []) if isinstance(v, str) and v.strip()), None) if "kelurahan_desa" in sub else None,
        "kecamatan": next((v for v in sub.get("kecamatan", []) if isinstance(v, str) and v.strip()), None) if "kecamatan" in sub else None,
        "kota_kabupaten": next((v for v in sub.get("kota_kabupaten", []) if isinstance(v, str) and v.strip()), None) if "kota_kabupaten" in sub else None,
        "provinsi": next((v for v in sub.get("provinsi", []) if isinstance(v, str) and v.strip()), None) if "provinsi" in sub else None,
        "alasan_viral_contoh": next((a for a in sub["alasan_viral"] if isinstance(a, str) and a.strip()), ""),
        "jumlah_artikel": len(sub),
        "jumlah_source_unik": sub["source"].nunique(),
        "earliest_article_date": dates.min() if not dates.empty else None,
        "latest_article_date": dates.max() if not dates.empty else None,
        "rentang_hari_pemberitaan": (dates.max() - dates.min()).days if len(dates) > 1 else 0,
        "explicit_viral_ratio": sub["is_explicitly_viral"].fillna(False).mean(),
        "indikasi_waktu_eksplisit": explicit_dates.iat[0] if not explicit_dates.empty else None,
        "avg_confidence": sub["confidence"].astype(float).mean() if "confidence" in sub else None,
        "contoh_link": sub["final_url"].iat[0],
    })

df_places = pd.DataFrame(agg_rows).sort_values("jumlah_source_unik", ascending=False).reset_index(drop=True)
print("Total tempat unik:", len(df_places))
df_places.head(10)

Total tempat unik: 1535


,canonical_group,nama_tempat,kategori,lokasi_text,alasan_viral_contoh,jumlah_artikel,jumlah_source_unik,earliest_article_date,latest_article_date,rentang_hari_pemberitaan,explicit_viral_ratio,indikasi_waktu_eksplisit,avg_confidence,contoh_link
0,914,Toko Kue Gambang,Kafe,"Pasar Johar Utara, Semarang","""Dirujak"" warganet hingga rating Google Maps anjlok (isu negatif)",9,8,2026-04-18 07:00:00+00:00,2026-08-12 07:00:00+00:00,116,1.000000,3 April 2026,0.705556,https://www.konteks.co.id/daerah/1632487841/toko-kue-gambang-semarang-diserbu-bintang-1-usai-viral-isu-anak-wali-kot...
1,229,Tahir Solo Museum,Museum,Kota Solo,Pengunjung mengeluhkan tarif parkir mobil ngepruk,7,7,2026-06-28 07:00:00+00:00,2026-09-21 22:52:00+00:00,85,0.285714,14 Agustus 2026,0.700000,https://regional.kompas.com/read/2026/09/22/055200178/satpam-di-pasar-triwindu-solo-viral-usai-tarik-parkir-ngepruk-...
2,821,Pasar Pagi Samarinda,lainnya,"Samarinda, Kalimantan Timur","Viral video sekuriti umpat wartawati ""media busuk"" (isu negatif)",7,7,2026-04-18 07:00:00+00:00,2026-09-07 07:00:00+00:00,142,0.857143,11 Agustus 2026,0.442857,https://regional.kompas.com/read/2026/08/12/151923278/sekuriti-pasar-pagi-umpat-media-busuk-ke-wartawati-wali-kota-s...
3,897,Pasar Baru,lainnya,Kota Kendari,Viral video temuan uang palsu di pasar (isu negatif),7,7,2026-04-02 07:00:00+00:00,2026-09-04 07:00:00+00:00,155,0.714286,April 2026,0.407143,https://suarasultra.com/2026/04/pedagang-pasar-baruga-kendari-resah-empat-lembar-uang-rp100-ribu-diduga-palsu-beredar/
4,470,Pantai Panjang,Wisata Alam,Kota Bengkulu,Mendapat bantuan lima gazebo gratis untuk wisatawan,6,5,2026-04-10 07:00:00+00:00,2026-09-22 14:15:44+00:00,165,0.000000,21 September 2026,0.641667,https://bengkulu.antaranews.com/berita/470141/himppera-serahkan-bantuan-gazebo-dukung-pengembangan-wisata-bengkulu
5,413,Museum dan Monumen Bahasa Nasional,Museum,"Pulau Penyengat, Kepulauan Riau","Pembangunan museum sejarah bahasa, calon destinasi wisata baru",8,5,2026-06-30 07:00:00+00:00,2026-09-25 11:53:51+00:00,87,0.000000,Groundbreaking 24 September 2026,0.787500,https://kepripedia.com/museum-dan-monumen-bahasa-nasional-dibangun-di-pulau-penyengat-bupati-bintan-kebanggaan-masya...
6,627,Pasar Ngasem,lainnya,"Jl. Polowijan No.11, Patehan, Kraton, Kota Yogyakarta",Surga kuliner tradisional selalu ramai,5,5,2026-05-16 07:00:00+00:00,2026-09-28 05:30:00+00:00,134,0.400000,April 2026,0.770000,https://www.tripzilla.id/pasar-kuliner-jogja/27629
7,526,Taman Margasatwa Ragunan,Wisata Alam,Jakarta Selatan,Masuk gratis HUT Jakarta ke-499,4,4,2026-05-03 07:00:00+00:00,2026-09-23 13:12:02+00:00,143,0.000000,"22, 27, 28 Juni 2026",0.637500,https://www.bloombergtechnoz.com/detail-news/113271/hut-ke-499-jakarta-masuk-tempat-wisata-gratis-cek-syaratnya/2
8,523,Wonderful Indonesia Tourism Fair (WITF) 2026,Pameran,"JIExpo Kemayoran, Jakarta",Pameran pariwisata B2B dengan 300+ buyers,4,4,2026-05-29 07:00:00+00:00,2026-09-26 01:25:00+00:00,119,0.000000,15-17 Oktober 2026,0.725000,https://travel.detik.com/travel-news/d-8676178/jakarta-jadi-titik-temu-pasar-dunia-dengan-destinasi-wisata-indonesia
9,434,Museum Sriwijaya Dharmakirti,Museum,Kawasan Cagar Budaya Nasional (KCBN) Muaro Jambi,"Museum baru dengan wahana imersif, resmi dibuka untuk umum",4,4,2026-04-17 07:00:00+00:00,2026-09-05 07:00:00+00:00,141,0.000000,29 Agustus 2026,0.825000,https://rri.co.id/jambi/regional/2692271/museum-sriwijaya-dharmakirti-muaro-jambi-di-buka-untuk-umum


In [14]:
# read hasil_ekstraksi_tempat_viral.xlsx, sheet Tempat
df_places_raw_location = pd.read_excel("hasil_ekstraksi_tempat_viral.xlsx", sheet_name="Tempat")
df_places_raw_location

,article_id,batch,nama_tempat,kategori,lokasi_text,kelurahan/desa,kecamatan,kota/kabupaten,provinsi,latitude,...,alasan_viral,indikasi_waktu,is_explicitly_viral,confidence,final_url (sumber isi),domain URL,judul (id_map),source (id_map),published (id_map),judul sesuai isi?
0,1,1,Soto Mie Agih,Restoran,"Jalan Suryakencana No.313, Babakan Pasar, Bogor Tengah, Bogor",Babakan Pasar,Bogor Tengah,Kota Bogor,Jawa Barat,-6.605,...,"Soto mie legendaris Bogor sejak 1965-an, menu babat hingga jumbo",NaN,Tidak,0.90,https://food.detik.com/rumah-makan/d-8679688/awas-kalap-ini-7-tempat-makan-soto-mie-legendaris-di-jakarta-dan-bogor,food.detik.com,Awas Kalap! Ini 7 Tempat Makan Soto Mie Legendaris di Jakarta dan Bogor - detikFood,detikFood,2026-09-26,Sesuai
1,1,1,Soto Mie Sarodja,Restoran,"Jalan Danau Tondano No.27, Pejompongan, Jakarta Pusat",Bendungan Hilir,Tanah Abang,Kota Jakarta Pusat,DKI Jakarta,-6.209,...,"Soto mie sejak 1970, diteruskan generasi kedua",NaN,Tidak,0.90,https://food.detik.com/rumah-makan/d-8679688/awas-kalap-ini-7-tempat-makan-soto-mie-legendaris-di-jakarta-dan-bogor,food.detik.com,Awas Kalap! Ini 7 Tempat Makan Soto Mie Legendaris di Jakarta dan Bogor - detikFood,detikFood,2026-09-26,Sesuai
2,1,1,Soto Mie Theresia,Restoran,"Jalan Gereja Theresia No.43, Menteng, Jakarta Pusat",Gondangdia,Menteng,Kota Jakarta Pusat,DKI Jakarta,-6.188,...,Soto mie legendaris sejak 1972 di kawasan Kuliner Theresia,NaN,Tidak,0.90,https://food.detik.com/rumah-makan/d-8679688/awas-kalap-ini-7-tempat-makan-soto-mie-legendaris-di-jakarta-dan-bogor,food.detik.com,Awas Kalap! Ini 7 Tempat Makan Soto Mie Legendaris di Jakarta dan Bogor - detikFood,detikFood,2026-09-26,Sesuai
3,1,1,Soto Mie Pak Fauzy,Restoran,"Kawasan Suryakencana, Bogor",NaN,Bogor Tengah,Kota Bogor,Jawa Barat,-6.598,...,Soto mie legendaris sejak 1974,NaN,Tidak,0.85,https://food.detik.com/rumah-makan/d-8679688/awas-kalap-ini-7-tempat-makan-soto-mie-legendaris-di-jakarta-dan-bogor,food.detik.com,Awas Kalap! Ini 7 Tempat Makan Soto Mie Legendaris di Jakarta dan Bogor - detikFood,detikFood,2026-09-26,Sesuai
4,2,1,Gudeg Yu Djum,Restoran,"Jl. Wijilan No. 167, Panembahan, Kraton, Yogyakarta",Panembahan,Kraton,Kota Yogyakarta,DI Yogyakarta,-7.806,...,"Ikon gudeg Jogja sejak 1950-an, harga ekonomis",NaN,Tidak,0.90,https://rri.co.id/yogyakarta/kuliner/2752639/rekomendasi-7-tempat-makan-ramah-kantong-di-jogja-dan-tips-menggunakan-...,rri.co.id,Rekomendasi 7 Tempat Makan Ramah Kantong di Jogja dan Tips Menggunakan GrabFood - RRI.co.id,RRI.co.id,2026-09-23,Sesuai
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1783,2168,73,Hazet,lainnya,Kota Tasikmalaya,NaN,NaN,Kota Tasikmalaya,Jawa Barat,-7.330,...,Kawasan pedestrian ikon kota yang dipromosikan lewat lagu Kongkow ke Hazet yang ramai di TikTok,NaN,Tidak,0.60,http://kilangbara.com/2026/08/12/lagu-kongkow-ke-hazet-gebrak-tiktok-netizen-gokil-ikut-karaoke-icon-kota-tasik-unju...,kilangbara.com,"Maru Bake House, Sensasi Nongkrong Bernuansa Gua Viral di Medan - suara usu",suara usu,2026-08-01,Tidak sesuai
1784,2179,73,SMOKIKI,Restoran,"Jl. Gurindam, Tlk. Tering, Kecamatan Batam Kota, Kota Batam",Teluk Tering,Batam Kota,Kota Batam,Kepulauan Riau,1.130,...,Restoran baru milik runner up MCI dengan BPK halal daging sapi,NaN,Tidak,0.75,https://metropolis.batampos.co.id/jelajah-kuliner-batam-5-tempat-makan-baru-dengan-menu-unik-dan-instagramable/,metropolis.batampos.co.id,"Diduga Belum Serah Terima, Pedagang 4 Pasar Dihimbau Bayar Kontibusi Kepada Wali Kota - Viral24.co.id",Viral24.co.id,2026-09-24,Tidak sesuai
1785,2179,73,Ruang Tamu Slow Bar Roastery,Kafe,"Jl. Orchid Park blok D no 136, Taman Baloi, Kecamatan Batam Kota, Kota Batam",Taman Baloi,Batam Kota,Kota Batam,Kepulauan Riau,1.130,...,"Kafe rumahan dengan kopi signature rempah, cocok WFC",NaN,Tidak,0.75,https://metropolis.batampos.co.id/jelajah-kuliner-batam-5-tempat-makan-baru-dengan-menu-unik-dan-instagramable/,metropolis.batampos.co.id,"Diduga Belum Serah

## 7. Gabungkan Data Lokasi Hasil Geocoding ke `df_places`

`df_places_raw_location` (dari `hasil_ekstraksi_tempat_viral.xlsx`, sheet `Tempat`) berisi data lokasi per baris tempat mentah (level artikel, sama seperti `df_places_raw`), termasuk hasil geocoding (`kelurahan/desa`, `kecamatan`, `kota/kabupaten`, `provinsi`, `latitude`, `longitude`, `presisi koordinat`, `sumber wilayah`, `catatan lokasi`).

Karena `df_places` sudah diagregasi per `canonical_group`, kolom lokasi digabung lewat dua langkah:
1. Cocokkan `df_places_raw_location` ke `df_places_raw` lewat `nama_tempat` + `final_url` (baris yang sama persis) untuk mendapatkan `canonical_group`-nya.
2. Ambil nilai lokasi pertama yang tidak kosong per `canonical_group`, lalu gabungkan ke `df_places`.

In [15]:
LOCATION_COLS = [
    "kelurahan/desa",
    "kecamatan",
    "kota/kabupaten",
    "provinsi",
    "latitude",
    "longitude",
    "presisi koordinat",
    "sumber wilayah",
    "catatan lokasi",
]

loc_merge = (
    df_places_raw_location
    .rename(columns={"final_url (sumber isi)": "final_url"})[["nama_tempat", "final_url"] + LOCATION_COLS]
    .drop_duplicates(subset=["nama_tempat", "final_url"])
)

df_places_raw_with_loc = df_places_raw.merge(loc_merge, on=["nama_tempat", "final_url"], how="left")


def first_valid(series):
    for v in series:
        if pd.notna(v) and str(v).strip():
            return v
    return None


loc_per_group = (
    df_places_raw_with_loc.groupby("canonical_group")[LOCATION_COLS]
    .agg(first_valid)
    .reset_index()
)

df_places = df_places.merge(loc_per_group, on="canonical_group", how="left")

print("Tempat dengan koordinat terisi:", df_places["latitude"].notna().sum(), "/", len(df_places))
df_places.head(10)

Tempat dengan koordinat terisi: 1517 / 1535


,canonical_group,nama_tempat,kategori,lokasi_text,alasan_viral_contoh,jumlah_artikel,jumlah_source_unik,earliest_article_date,latest_article_date,rentang_hari_pemberitaan,...,contoh_link,kelurahan/desa,kecamatan,kota/kabupaten,provinsi,latitude,longitude,presisi koordinat,sumber wilayah,catatan lokasi
0,914,Toko Kue Gambang,Kafe,"Pasar Johar Utara, Semarang","""Dirujak"" warganet hingga rating Google Maps anjlok (isu negatif)",9,8,2026-04-18 07:00:00+00:00,2026-08-12 07:00:00+00:00,116,...,https://www.konteks.co.id/daerah/1632487841/toko-kue-gambang-semarang-diserbu-bintang-1-usai-viral-isu-anak-wali-kot...,Kauman,Semarang Tengah,Kota Semarang,Jawa Tengah,-6.9680,110.4240,Pusat kelurahan/desa,teks+inferensi,Pasar Johar
1,229,Tahir Solo Museum,Museum,Kota Solo,Pengunjung mengeluhkan tarif parkir mobil ngepruk,7,7,2026-06-28 07:00:00+00:00,2026-09-21 22:52:00+00:00,85,...,https://regional.kompas.com/read/2026/09/22/055200178/satpam-di-pasar-triwindu-solo-viral-usai-tarik-parkir-ngepruk-...,None,Jebres,Kota Surakarta,Jawa Tengah,-7.5670,110.8170,Pusat kota/kabupaten,teks+inferensi,Dekat kampus UNS
2,821,Pasar Pagi Samarinda,lainnya,"Samarinda, Kalimantan Timur","Viral video sekuriti umpat wartawati ""media busuk"" (isu negatif)",7,7,2026-04-18 07:00:00+00:00,2026-09-07 07:00:00+00:00,142,...,https://regional.kompas.com/read/2026/08/12/151923278/sekuriti-pasar-pagi-umpat-media-busuk-ke-wartawati-wali-kota-s...,Pasar Pagi,Samarinda Kota,Kota Samarinda,Kalimantan Timur,-0.5030,117.1480,Pusat kelurahan/desa,teks+inferensi,None
3,897,Pasar Baru,lainnya,Kota Kendari,Viral video temuan uang palsu di pasar (isu negatif),7,7,2026-04-02 07:00:00+00:00,2026-09-04 07:00:00+00:00,155,...,https://suarasultra.com/2026/04/pedagang-pasar-baruga-kendari-resah-empat-lembar-uang-rp100-ribu-diduga-palsu-beredar/,Pasar Baru,Baruga,Kota Kendari,Sulawesi Tenggara,-4.0400,122.5100,Pusat kecamatan,teks+inferensi,None
4,470,Pantai Panjang,Wisata Alam,Kota Bengkulu,Mendapat bantuan lima gazebo gratis untuk wisatawan,6,5,2026-04-10 07:00:00+00:00,2026-09-22 14:15:44+00:00,165,...,https://bengkulu.antaranews.com/berita/470141/himppera-serahkan-bantuan-gazebo-dukung-pengembangan-wisata-bengkulu,None,None,Kota Bengkulu,Bengkulu,-3.8100,102.2700,Titik tempat,teks+inferensi,Pantai memanjang melintasi beberapa kecamatan
5,413,Museum dan Monumen Bahasa Nasional,Museum,"Pulau Penyengat, Kepulauan Riau","Pembangunan museum sejarah bahasa, calon destinasi wisata baru",8,5,2026-06-30 07:00:00+00:00,2026-09-25 11:53:51+00:00,87,...,https://kepripedia.com/museum-dan-monumen-bahasa-nasional-dibangun-di-pulau-penyengat-bupati-bintan-kebanggaan-masya...,Penyengat,Tanjungpinang Kota,Kota Tanjungpinang,Kepulauan Riau,0.9270,104.4190,Pusat kelurahan/desa,teks+inferensi,None
6,627,Pasar Ngasem,lainnya,"Jl. Polowijan No.11, Patehan, Kraton, Kota Yogyakarta",Surga kuliner tradisional selalu ramai,5,5,2026-05-16 07:00:00+00:00,2026-09-28 05:30:00+00:00,134,...,https://www.tripzilla.id/pasar-kuliner-jogja/27629,Patehan,Kraton,Kota Yogyakarta,DI Yogyakarta,-7.8095,110.3605,Titik tempat,teks+inferensi,"Pasar Ngasem berada di Patehan, Kraton; teks menyebut kompleks Taman Budaya"
7,526,Taman Margasatwa Ragunan,Wisata Alam,Jakarta Selatan,Masuk gratis HUT Jakarta ke-499,4,4,2026-05-03 07:00:00+00:00,2026-09-23 13:12:02+00:00,143,...,https://www.bloombergtechnoz.com/detail-news/113271/hut-ke-499-jakarta-masuk-tempat-wisata-gratis-cek-syaratnya/2,Ragunan,Pasar Minggu,Kota Jakarta Selatan,DKI Jakarta,-6.3125,106.8200,Titik tempat,teks+inferensi,None
8,523,Wonderful Indonesia Tourism Fair (WITF) 2026,Pameran,"JIExpo Kemayoran, Jakarta",Pameran pariwisata B2B dengan 300+ buyers,4,4,2026-05-29 07:00:00+00:00,2026-09-26 01:25:00+00:00,119,...,https://travel.detik.com/travel-news/d-8676178/jakarta-jadi-titik-temu-pasar-dunia-dengan-destinasi-wisata-indonesia,Pademangan Timur,Pademangan,Kota Jakarta Utara,DKI Jakarta,-6.1470,106.8460,Titik tempat,teks+inferensi,"JIExpo secara administratif di Pa

In [18]:
df_places.to_csv("df_places_29092026.csv")